<a href="https://colab.research.google.com/github/shuvankar09/Python_assingment_pw/blob/main/Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np

In [17]:
# Load dataset
df = pd.read_csv("/content/healthcare_data_cleaning_dataset.csv")

In [18]:
# Q1. Missing Data Identification

# Check missing values
missing_values = df.isnull().sum()

# Calculate percentage of missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100

# Create summary table
missing_data = pd.DataFrame({
    "Missing Values": missing_values,
    "Percentage": missing_percentage
})

print(missing_data)

                    Missing Values  Percentage
Patient_ID                       0    0.000000
Age                            600   11.764706
Gender                           0    0.000000
City                             0    0.000000
Diagnosis                        0    0.000000
Hospital_Visits                  0    0.000000
Treatment_Cost                 593   11.627451
Insurance_Coverage               0    0.000000
Admission_Date                   0    0.000000


In [19]:
# Q2. Handling Missing Age

# Check missing values in Age column
print(df["Age"].isnull().sum())

# Replace missing Age values with median
median_age = df["Age"].median()
df["Age"].fillna(median_age, inplace=True)

600


/tmp/ipykernel_951/379066999.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Age"].fillna(median_age, inplace=True)


In [20]:
# Q3. Handling Missing Treatment Cost

# Check missing values in Treatment_Cost
print(df["Treatment_Cost"].isnull().sum())

# Replace missing Treatment_Cost values with median
median_cost = df["Treatment_Cost"].median()

df["Treatment_Cost"].fillna(median_cost, inplace=True)

# Verify again
print(df["Treatment_Cost"].isnull().sum())

593
0


/tmp/ipykernel_951/1009182128.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Treatment_Cost"].fillna(median_cost, inplace=True)


In [21]:
# Q4. Duplicate Patient Records

# Dataset size before removing duplicates
before_rows = df.shape[0]

print("Rows before removing duplicates:", before_rows)

# Identify duplicate rows
duplicates = df[df.duplicated()]

print("Duplicate Rows:")
print(duplicates)

# Remove duplicate rows
df = df.drop_duplicates()

# Dataset size after removing duplicates
after_rows = df.shape[0]

print("Rows after removing duplicates:", after_rows)

# Compare dataset size
print("Total duplicates removed:", before_rows - after_rows)

Rows before removing duplicates: 5100
Duplicate Rows:
      Patient_ID   Age  Gender       City     Diagnosis  Hospital_Visits  \
5000       15126  50.0    Male  Hyderabad        Asthma                1   
5001       15108  50.0    Male  Hyderabad        Asthma                3   
5002       10441  50.0    Male    Chennai           Flu                3   
5003       15975  50.0  Female  Hyderabad           Flu               10   
5004       18427  50.0  Female      Delhi      COVID-19               13   
...          ...   ...     ...        ...           ...              ...   
5094       11571  50.0  Female    Chennai      COVID-19                4   
5096       17597  50.0  Female    Chennai        Asthma                2   
5097       19171  50.0  Female     Mumbai           Flu                1   
5098       13854  50.0  Female  Bangalore           Flu               17   
5099       14859  50.0    Male      Delhi  Hypertension               19   

      Treatment_Cost  Insurance_C

In [22]:
# Q5. Invalid Age Values (Data Quality Check)

# Detect invalid age values (<0 or >100)
invalid_age = df[(df["Age"] < 0) | (df["Age"] > 100)]

print("Invalid Age Records:")
print(invalid_age)

# Count invalid records
print("Number of invalid age values:", invalid_age.shape[0])

# Option 1: Remove invalid records
df = df[(df["Age"] >= 0) & (df["Age"] <= 100)]

Invalid Age Records:
Empty DataFrame
Columns: [Patient_ID, Age, Gender, City, Diagnosis, Hospital_Visits, Treatment_Cost, Insurance_Coverage, Admission_Date]
Index: []
Number of invalid age values: 0


In [23]:
# Q6. Outlier Detection (Treatment Cost)

# Calculate Q1 and Q3
Q1 = df["Treatment_Cost"].quantile(0.25)
Q3 = df["Treatment_Cost"].quantile(0.75)

# Interquartile Range (IQR)
IQR = Q3 - Q1

# Define outlier boundaries
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Detect outliers
outliers = df[(df["Treatment_Cost"] < lower_bound) | (df["Treatment_Cost"] > upper_bound)]

print("Number of outliers in Treatment_Cost:", outliers.shape[0])

Number of outliers in Treatment_Cost: 50


In [24]:
# Q7. Outlier Treatment

# Calculate 5th and 95th percentile
lower_percentile = df["Treatment_Cost"].quantile(0.05)
upper_percentile = df["Treatment_Cost"].quantile(0.95)

# Apply capping (Winsorization)
df["Treatment_Cost_Capped"] = df["Treatment_Cost"].copy()

df["Treatment_Cost_Capped"] = df["Treatment_Cost_Capped"].apply(
    lambda x: lower_percentile if x < lower_percentile
    else upper_percentile if x > upper_percentile
    else x
)

# Check result
print(df[["Treatment_Cost", "Treatment_Cost_Capped"]].head())

   Treatment_Cost  Treatment_Cost_Capped
0         41010.0                41010.0
1         12194.0                12194.0
2         45086.0                45086.0
3         40842.0                40842.0
4          9873.0                 9873.0


In [25]:
# 8. Transformation


# Create new column with log transformation
df["Treatment_Cost_Log"] = np.log1p(df["Treatment_Cost"])
# log1p = log(1 + x) to handle zero values safely

# Compare original vs transformed values
print(df[["Treatment_Cost", "Treatment_Cost_Log"]].head())

   Treatment_Cost  Treatment_Cost_Log
0         41010.0           10.621596
1         12194.0            9.408781
2         45086.0           10.716349
3         40842.0           10.617491
4          9873.0            9.197660


In [26]:
# Q9. Time-Based Missing Handling

import pandas as pd

# Ensure Admission_Date is in datetime format
df["Admission_Date"] = pd.to_datetime(df["Admission_Date"])

# Sort data by Admission_Date
df = df.sort_values(by="Admission_Date")

# Apply forward fill (or backward fill if needed)
df["Admission_Date"] = df["Admission_Date"].ffill()   # forward fill
# Alternative:
# df["Admission_Date"] = df["Admission_Date"].bfill()  # backward fill

# Check missing values
print(df["Admission_Date"].isnull().sum())

# Show first few rows to confirm changes
print(df.head(10))

0
      Patient_ID   Age  Gender       City     Diagnosis  Hospital_Visits  \
2932       19591  91.0    Male    Chennai        Asthma                3   
4617       10896  88.0  Female    Chennai      COVID-19                3   
2642       19262  50.0  Female  Hyderabad        Asthma                9   
4837       12843  89.0  Female  Bangalore      COVID-19               14   
2702       17230  41.0    Male  Bangalore  Hypertension               18   
329        13863  61.0  Female      Delhi      Diabetes                1   
348        18716  81.0    Male      Delhi           Flu                4   
4563       19175  12.0  Female      Delhi           Flu                3   
1551       10959  17.0    Male    Chennai  Hypertension                8   
1791       14612  94.0    Male      Delhi  Hypertension               14   

      Treatment_Cost  Insurance_Coverage Admission_Date  \
2932    32149.000000                   0     2023-01-01   
4617   199702.965333                   1   